In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/AI-Powered Hospitality Revenue Optimization/Dataset/"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(base_path + "hotel_bookings_cleaned.csv")

In [ ]:
features = [
    "adr",
    "total_stay_nights",
    "lead_time",
    "total_guests",
    "total_of_special_requests"
]

X = df[features]

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.02,
    random_state=42
)

df["anomaly"] = model.fit_predict(X)

In [ ]:
df["anomaly_flag"] = df["anomaly"].map({
    1: "Normal",
    -1: "Anomaly"
})

In [ ]:
print(df["anomaly_flag"].value_counts())

anomaly_flag
Normal     85485
Anomaly     1745
Name: count, dtype: int64


In [ ]:
print(
    df["anomaly_flag"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

anomaly_flag
Normal     98.0
Anomaly     2.0
Name: proportion, dtype: float64


In [ ]:
anomalies = df[df["anomaly_flag"] == "Anomaly"]

anomalies[
    [
        "hotel",
        "arrival_date",
        "adr",
        "total_stay_nights",
        "lead_time",
        "total_guests",
        "total_of_special_requests",
        "estimated_revenue"
    ]
].head(20)

,hotel,arrival_date,adr,total_stay_nights,lead_time,total_guests,total_of_special_requests,estimated_revenue
0,Resort Hotel,2015-07-01,0.00,0,342,2.0,0,0.00
1,Resort Hotel,2015-07-01,0.00,0,737,2.0,0,0.00
28,Resort Hotel,2015-07-01,62.00,14,118,1.0,2,868.00
82,Resort Hotel,2015-07-03,134.73,14,324,2.0,0,1886.22
84,Resort Hotel,2015-07-03,108.73,21,79,3.0,2,2283.33
224,Resort Hotel,2015-07-09,123.00,1,17,4.0,3,123.00
256,Resort Hotel,2015-07-10,165.00,5,21,4.0,3,825.00
320,Resort Hotel,2015-07-12,133.16,14,55,12.0,1,1864.24
386,Resort Hotel,2015-07-14,230.67,6,317,4.0,2,1384.02
396,Resort Hotel,2015-07-14,187.50,15,113,3.0,3,2812.50


In [ ]:
importance = pd.DataFrame({
    "Feature": features
})

importance

,Feature
0,adr
1,total_stay_nights
2,lead_time
3,total_guests
4,total_of_special_requests


In [ ]:
comparison = df.groupby("anomaly_flag")[features].mean()
comparison.round(2)

,adr,total_stay_nights,lead_time,total_guests,total_of_special_requests
anomaly_flag,,,,,
Anomaly,170.76,8.26,163.14,3.21,1.52
Normal,105.21,3.53,78.27,2.00,0.68


In [ ]:
df.groupby("anomaly_flag")["estimated_revenue"].agg(
    ["count", "mean", "median", "sum"]
).round(2)

,count,mean,median,sum
anomaly_flag,,,,
Anomaly,1745,1197.63,1039.5,2089858.16
Normal,85485,378.57,296.0,32361796.77


In [ ]:
anomaly_summary = (
    df[df["anomaly_flag"] == "Anomaly"]
    .groupby("hotel")
    .agg(
        anomalies=("anomaly_flag", "size"),
        avg_adr=("adr", "mean"),
        avg_stay=("total_stay_nights", "mean"),
        avg_lead_time=("lead_time", "mean"),
        revenue=("estimated_revenue", "sum")
    )
    .reset_index()
)

anomaly_summary.round(2)

,hotel,anomalies,avg_adr,avg_stay,avg_lead_time,revenue
0,City Hotel,627,159.56,6.03,140.36,559376.43
1,Resort Hotel,1118,177.04,9.52,175.91,1530481.73
